# LSTM example without predictors (economist)
This dataset was produced from US economic time series data available from https://fred.stlouisfed.org/.

Variables:

1. date - Month of data collection
2. pce - personal consumption expenditures, in billions of dollars, https://fred.stlouisfed.org/series/PCE
3. pop - total population, in thousands, https://fred.stlouisfed.org/series/POP
4. psavert - personal savings rate, https://fred.stlouisfed.org/series/PSAVERT/
5. uempmed - median duration of unemployment, in weeks, https://fred.stlouisfed.org/series/UEMPMED
6. unemploy - number of unemployed in thousands, https://fred.stlouisfed.org/series/UNEMPLOY

We want to predict the unemployment in the following 12 months (*unemploy* variable).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import LSTM, Dropout, TimeDistributed, Dense, Input

from sklearn.preprocessing import StandardScaler

# 1. Load the dataset
df_passengers = pd.read_csv('air_passengers.csv', index_col='Month', parse_dates=True)
df_passengers.index.freq = 'MS' # Set frequency to Month Start
df_passengers['Passengers'] = df_passengers['International airline passengers: monthly totals in thousands.']
print("Original DataFrame head:")
print(df_passengers.head())

# Plotting the original time series
plt.figure(figsize=(12, 6))
plt.plot(df_passengers.index, df_passengers['Passengers'], label='Original Passengers Data')
plt.title('International Airline Passengers (1949-1960)')
plt.xlabel('Date')
plt.ylabel('Passengers (thousands)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 2. Preprocessing: Scaling data
scaler_passengers = StandardScaler()
df_passengers['Passengers_Scaled'] = scaler_passengers.fit_transform(df_passengers[['Passengers']])

print("Scaled DataFrame head:")
print(df_passengers.head())

# 3. Prepare Tensors for LSTM
prediction_months = 12
lag_months = prediction_months
features_count = 1

# Split data into training and test sets
# Training data is all but the last 'prediction_months' values
df_train_passengers = df_passengers[:-prediction_months].copy()

# Test data is the last 'prediction_months' values (actuals for comparison)
df_test_actual_passengers = df_passengers[-prediction_months:].copy()

# Create lagged data for X_train
# The input for training will be 'lag_months' values, predicting the next 'lag_months' values
df_train_x_lag_passengers = np.array([df_train_passengers['Passengers_Scaled'].values[i:(i + lag_months)]
                                      for i in range(len(df_train_passengers) - lag_months - prediction_months + 1)])

# Convert to tensor format (samples, timesteps, features)
df_train_x_tensor_passengers = df_train_x_lag_passengers.reshape(
    (df_train_x_lag_passengers.shape[0], lag_months, features_count))

# Create lagged data for Y_train (target is the sequence of next 'lag_months' values)
df_train_y_lag_passengers = np.array([df_train_passengers['Passengers_Scaled'].values[(i + 1):(i + lag_months + 1)]
                                      for i in range(len(df_train_passengers) - lag_months - prediction_months + 1)])

# Convert to tensor format
df_train_y_tensor_passengers = df_train_y_lag_passengers.reshape(
    (df_train_y_lag_passengers.shape[0], lag_months, features_count))

print(f"Shape of df_train_x_tensor_passengers: {df_train_x_tensor_passengers.shape}")
print(f"Shape of df_train_y_tensor_passengers: {df_train_y_tensor_passengers.shape}")

# Prepare input for actual prediction (last 'lag_months' observations from the training set)
# We take the last 'lag_months' values from the entire dataset, before the actual test set starts
df_predict_input = df_passengers['Passengers_Scaled'].values[-(prediction_months + lag_months):-prediction_months]

# Reshape for LSTM prediction (1 sample, lag_months timesteps, 1 feature)
df_predict_tensor = df_predict_input.reshape((1, lag_months, features_count))

print(f"Shape of df_predict_tensor: {df_predict_tensor.shape}")

In [ ]:
# 4. Define LSTM Model
lstm_model_passengers = Sequential([
    Input(batch_shape=(1, lag_months, features_count)), # 1 - batch size, lag_months - timesteps, 1 - feature
    LSTM(units=50, return_sequences=True, stateful=True),
    Dropout(rate=0.25),
    LSTM(units=50, return_sequences=True, stateful=True),
    Dropout(rate=0.25),
    TimeDistributed(Dense(units=1)) # TimeDistributed layer to apply a dense layer to each timestep
])

lstm_model_passengers.summary()

In [ ]:
# 5. Compile the model
lstm_model_passengers.compile(
    loss='mean_absolute_error',
    optimizer=Adam(),
    metrics=['mean_absolute_percentage_error', 'mean_squared_error']
)

# 6. Train the model
history_passengers = lstm_model_passengers.fit(
    x=df_train_x_tensor_passengers,
    y=df_train_y_tensor_passengers,
    batch_size=1,       # Each batch contains 1 sample
    epochs=20,          # Number of epochs
    verbose=1,          # Verbosity mode
    shuffle=False       # Maintain the order of the dataset
)

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(history_passengers.history['loss'], label='Loss (MAE)')
plt.xlabel('Epoch')
plt.ylabel('Mean Absolute Error')
plt.title('LSTM Training Loss over Epochs (Passengers)')
plt.legend()
plt.grid(True)
plt.show()

# 7. Make predictions
# Reset states before prediction, if stateful model is used for sequential predictions.
# For a single prediction of a sequence, it's not strictly necessary to reset, but good practice if it was trained batch by batch.
# lstm_model_passengers.reset_states()

lstm_forecast_passengers_scaled = lstm_model_passengers.predict(df_predict_tensor, batch_size=1)
lstm_forecast_passengers_scaled = lstm_forecast_passengers_scaled.flatten()

# Rescale the forecast back to original values
lstm_forecast_passengers = scaler_passengers.inverse_transform(lstm_forecast_passengers_scaled.reshape(-1, 1)).flatten()

# Create DataFrame for forecast
forecast_dates = pd.date_range(start=df_passengers.index[-prediction_months], periods=prediction_months, freq='MS')
df_forecast_passengers = pd.DataFrame({
    'Month': forecast_dates,
    'Passengers': lstm_forecast_passengers
})
df_forecast_passengers.set_index('Month', inplace=True)

print("LSTM Forecast (last 12 months):")
print(df_forecast_passengers)

# 8. Visualize the results
plt.figure(figsize=(14, 7))
plt.plot(df_train_passengers.index, scaler_passengers.inverse_transform(df_train_passengers[['Passengers_Scaled']]), label='Training Data (Actual)', color='blue')
plt.plot(df_test_actual_passengers.index, df_test_actual_passengers['Passengers'], label='Actual Test Data', color='orange')
plt.plot(df_forecast_passengers.index, df_forecast_passengers['Passengers'], label='LSTM Forecast', color='red', linestyle='--')

plt.title('International Airline Passengers: Actual vs. LSTM Forecast (Last 12 Months)')
plt.xlabel('Date')
plt.ylabel('Passengers (thousands)')
plt.legend()
plt.grid(True)
plt.show()